# RUN ON COLAB (free T4 GPU)

Before running: **Runtime > Change runtime type > T4 GPU**.

This notebook is a thin wrapper — it doesn't contain any training logic itself.
All it does is: mount your data, install this repo, and call `training/train.py`
exactly as you would from a terminal. If you want to understand or change the
actual training loop, edit `training/train.py`, not this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Expect a zip of data_gen's output (images/, masks/, metadata.jsonl, splits.json)
# uploaded to this path — produced by `python -m data_gen.generate_dataset` locally.
DATASET_ZIP = '/content/drive/MyDrive/sanding-region-segmentation/dataset.zip'
!mkdir -p /content/data
!unzip -q -o "$DATASET_ZIP" -d /content/data

In [ ]:
!git clone https://github.com/DharshanaReddy/sanding-region-segmentation.git /content/repo
%cd /content/repo
!pip install -q -e ".[train]"

In [ ]:
!python -m training.train \
  --data-dir /content/data \
  --model deeplabv3_mobilenet \
  --epochs 30 \
  --batch-size 16 \
  --image-size 512 \
  --output-dir /content/checkpoints \
  --log-dir /content/runs

In [ ]:
# Save checkpoints back to Drive so a disconnected Colab session doesn't lose them.
!mkdir -p /content/drive/MyDrive/sanding-region-segmentation/checkpoints
!cp /content/checkpoints/*.pt /content/checkpoints/*.json /content/drive/MyDrive/sanding-region-segmentation/checkpoints/

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs